## AMCL Performance Analysis

In [43]:
import math
import numpy as np
import pandas as pd
from rosbags.rosbag2 import Reader
import scipy.io as sio
"""
Analysis Outline
Collected Topics
- /amcl_pose, /particle_cloud, /odometry/filtered
Scenerio
- took 2 turns around 3 waypoints, stopping for 5 seconds at each waypoint after the waypoint is reached

Waypoint coordinates relative to the map frame in detail

w1->(1.18, 2.52, pi/2)
w2->(-0.75, 2.57, pi/2)
w3->(0,0,pi/2)

Tasks
->Capture the arrival times for all the 6 points
->store /amcl_pose + /odometry/filtered estimates for all the points / (x,y,theta).
->calculate statistics: mean absolute error, mean_squared_error, estimates
"""

GROUND_TRUTHS = {
    "w1": (1.18, 2.52, math.pi/2),
    "w2": (-0.75, 2.57, math.pi/2),
    "w3": ((0, 0, math.pi/2))
}

# --- CONFIGURATION ---
BAG_PATH = "/home/rosgeek/Documents/lab_parkur_ws/src/lab_mapping/rosbags/rosbag2_2026_04_20-07_28_15"  # Folder containing the .db3
TOPIC = "/amcl_pose"

In [44]:
from pathlib import Path
from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore
import datetime

class Pose:
     @classmethod
     def get_average(self, poses):
          n = len(poses)
          sum_pose = Pose()
          for pose in poses:
               sum_pose.x+=pose.x
               sum_pose.y+=pose.y
               sum_pose.theta+=pose.theta
          sum_pose.x /= n
          sum_pose.y /= n
          sum_pose.theta /= n

          return sum_pose

     def __init__(self, x=0.0,y=0.0,theta=0.0,time=0.0):
          self.x = x
          self.y = y
          self.theta = theta
          self.time = time
          
     def __repr__(self):
          return f"x: {self.x}, y: {self.y}, theta: {self.theta}, time: {self.time}"
     
def quaternion_to_yaw(qx, qy, qz, qw):
    """Converts 2D planar quaternion to Euler Yaw (theta) in radians."""
    siny_cosp = 2 * (qw * qz + qx * qy)
    cosy_cosp = 1 - 2 * (qy * qy + qz * qz)
    return math.atan2(siny_cosp, cosy_cosp)

def timestamp_to_hour_minute(timestamp):
     return datetime.datetime.fromtimestamp(timestamp).strftime('%H:%M:%S')

In [45]:
bagpath = Path(BAG_PATH)
typestore = get_typestore(Stores.ROS2_FOXY)

amcl_poses = [] #Pose array
odom_filtered_poses = []
scan_data = []
scan_timestamps = []
# Create reader instance and open for reading.
with AnyReader([bagpath], default_typestore=typestore) as reader:
    connections = [x for x in reader.connections if x.topic == '/amcl_pose']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         yaw = quaternion_to_yaw(msg.pose.pose.orientation.x, 
                                 msg.pose.pose.orientation.y, 
                                 msg.pose.pose.orientation.z, 
                                 msg.pose.pose.orientation.w)
         x,y = msg.pose.pose.position.x, msg.pose.pose.position.y
         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         pose = Pose(x,y,yaw,time)
         amcl_poses.append(pose)

    connections = [x for x in reader.connections if x.topic == '/odometry/filtered']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         yaw = quaternion_to_yaw(msg.pose.pose.orientation.x, 
                                 msg.pose.pose.orientation.y, 
                                 msg.pose.pose.orientation.z, 
                                 msg.pose.pose.orientation.w)
         x,y = msg.pose.pose.position.x, msg.pose.pose.position.y
         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         odom_filtered_poses.append((x,y,yaw,time))

    connections = [x for x in reader.connections if x.topic == '/scan']
    params_printed = False
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         if not params_printed:
              print(f"angle_min: {msg.angle_min}, angle_max: {msg.angle_max}, angle_increment: {msg.angle_increment}, range_min: {msg.range_min}, range_max: {msg.range_max}")
              params_printed = True

         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         scan_data.append(msg.ranges)
         scan_timestamps.append(time)
     

angle_min: -3.1415927410125732, angle_max: 3.1415927410125732, angle_increment: 0.008738784119486809, range_min: 0.05000000074505806, range_max: 18.0


In [22]:
pose_data = pd.DataFrame(columns=["x_vals", "y_vals", "theta_vals", "unix_time"], data=np.array(odom_filtered_poses))

In [32]:
pose_data.head()

,x_vals,y_vals,theta_vals,unix_time
0,0.0,0.0,-0.001583,1.776670e+09
1,0.0,0.0,-0.001584,1.776670e+09
2,0.0,0.0,-0.001585,1.776670e+09
3,0.0,0.0,-0.001584,1.776670e+09
4,0.0,0.0,-0.001585,1.776670e+09


In [28]:
pose_data["theta_vals"].mean()

-0.21521923375008567

In [33]:
pose_data.to_csv("poses.csv")

In [ ]:
scan_arr = np.array(scan_data)
scan_arr.shape

(3838, 720)

In [51]:
sio.savemat("laser_scan.mat", {"scan_array": scan_data, "scan_times": scan_timestamps})
sio.savemat("poses_data.mat", {"poses": pose_data})